In [ ]:
#1:
#this restriction introduces survivorship bias by making the DOW Jones (or any other index)
#look like it's doing better than perhaps it historically has. Because we invest based on historical data,
#our risk and return values are both skewed.

#2:
# first we borrow code from the chapter to compute our efficient frontier

!pip install -q tidyfinance
!pip install -q adjustText

import sys
print(sys.version)

import pandas as pd
import numpy as np
import tidyfinance as tf

from plotnine import *
from mizani.formatters import percent_format
from adjustText import adjust_text

symbols = tf.download_data(
    domain = "constituents",
    index = "Dow Jones Industrial Average"
)

prices_daily = tf.download_data(
    domain = 'stock_prices',
    symbols = symbols['symbol'].tolist(),
    start_date = '2000-01-01',
    end_date = '2023-12-31'
)

prices_daily = (prices_daily
    .groupby('symbol')
    .apply(lambda x: x.assign(counts=x['adjusted_close'].dropna().count()))
    .reset_index(drop=True)
    .query('counts == counts.max()')
)

returns = (
    prices_daily
    .assign(
        date = lambda x: x['date'].dt.to_period('M').dt.to_timestamp()
    )
    .groupby(['symbol','date'], as_index = False)
    .agg(adjusted_close=('adjusted_close','last'))
    .assign(ret=lambda x: x.groupby('symbol')['adjusted_close'].pct_change())
)

assets = (
    returns
    .groupby('symbol', as_index = False)
    .agg(
        mu = ('ret','mean'),
        sigma = ('ret','std')
    )
    .assign(
        mu_annualized = lambda x: (1+x['mu'])**12 - 1,
        sigma_annualized = lambda x: x['sigma']*np.sqrt(12)
    )
)

print(assets)

#now we mutliply the average monthly returns by 12 to annualize
#then we mutliply the std of the monthly returns by sqrt(12) to find the volatility of annual returns
#note that mu is now in terms of annual expected returns

assets_figure = (
    ggplot(
        assets, aes(x='sigma_annualized',y='mu_annualized',label = 'symbol')
    )
    + geom_point()
    + geom_text(adjust_text={'arrowprops': {'arrowstyle': '-'}})
    + scale_x_continuous(labels=percent_format())
    + scale_y_continuous(labels=percent_format())
    + labs(
        x='Volatility', y='Expected Return',
        title='Expected Return and Volatility of DOW Jones Index constituents'
    )
)

returns_wide = (
    returns
    .pivot(index='date', columns='symbol', values='ret')
    .reset_index()
)

sigma = (
    returns_wide
    .drop(columns=['date'])
    .cov() * 12 #this also must be annualized now
)

sigma_long = (
    sigma
    .reset_index()
    .melt(id_vars='symbol', var_name='symbol_b', value_name='value')
)

sigma_long['symbol_b'] = pd.Categorical(
    sigma_long['symbol_b'],
    categories=sigma_long['symbol_b'].unique()[::-1],
    ordered = True
)

sigma_figure = (
    ggplot(
        sigma_long,
        aes(x='symbol',y='symbol_b',fill='value')
    )
    + geom_tile()
    + labs(x='',y='',fill='(Co-)Variance', title='Sample Variance-Covariance Matrix of Dow Jones index Constituents')
    + scale_fill_continuous(labels=percent_format())
    + theme(axis_text_x=element_text(angle=45, hjust=1))
)

iota = np.ones(sigma.shape[0])
sigma_inv = np.linalg.inv(sigma.values)
omega_mvp = (sigma_inv @ iota) / (iota @ sigma_inv @ iota)

assets = assets.assign(omega_mvp=omega_mvp)

assets['symbol'] = pd.Categorical(
    assets['symbol'],
    categories = assets.sort_values('omega_mvp')['symbol'],
    ordered = True
)

omega_figure = (
    ggplot(
        assets, aes(y='omega_mvp', x='symbol', fill='omega_mvp>0')
    )
    + geom_col()
    + coord_flip()
    + scale_y_continuous(labels=percent_format())
    + labs(x='',y='', title='Minimum-variance portfolio weights')
    + theme(legend_position='none')
)
omega_figure.show()

mu = assets['mu_annualized'].values
mu_mvp = omega_mvp @ mu
sigma_mvp = np.sqrt(omega_mvp @ sigma.values @ omega_mvp)
summary_mvp = pd.DataFrame(
    {
        'mu_annualized': [mu_mvp],
        'sigma_annualized': [sigma_mvp],
        'type': ['Minimum-variance Portfolio'],
    }
)

mu_bar = assets['mu_annualized'].max()

C = iota @ sigma_inv @ iota
D = iota @ sigma_inv @ mu
E = mu @ sigma_inv @ mu
lambda_tilde = 2 * (mu_bar - D / C) / (E - (D**2) / C)
omega_efp = omega_mvp + (lambda_tilde / 2) * (sigma_inv @ mu - D * omega_mvp)
mu_efp = omega_efp @ mu
sigma_efp = np.sqrt(omega_efp @ sigma.values @ omega_efp)

summary_efp = pd.DataFrame(
    {'mu_annualized': [mu_efp], 'sigma_annualized': [sigma_efp], 'type': ['Efficient Portfolio']}
)

summaries = pd.concat([assets, summary_mvp, summary_efp], ignore_index = True)

summaries_figure = (
    ggplot(summaries, aes(x='sigma_annualized',y='mu_annualized'))
    + geom_point(data=summaries.query('type.isna()'))
    + geom_point(data=summaries.query('type.notna()'), color='#F21A00', size=3)
    + geom_label(
        aes(label='type'), adjust_text={'arrowprops': {'arrowstyle': '-'}}
    )
    + scale_x_continuous(labels=percent_format())
    + scale_y_continuous(labels=percent_format())
    + labs(x = 'Volatility', y = 'Expected Return', title='Efficient and minimum-variance Portfolios')
)

efficient_frontier = (
    pd.DataFrame({'a': np.arange(-1,2.01,0.01).round(2)})
    .assign(
        omega=lambda x: x['a'].map(
            lambda x: x * omega_efp + (1-x) * omega_mvp
        )
    )
    .assign(
        mu_annualized=lambda x: x['omega'].map(lambda x: x @ mu),
        sigma_annualized=lambda x: x['omega'].map(lambda x: np.sqrt(x @ sigma @ x))
    )
)

summaries = pd.concat([summaries, efficient_frontier], ignore_index=True)

summaries_figure = (
    ggplot(summaries, aes(x='sigma_annualized',y='mu_annualized'))
    + geom_point(data=summaries.query('type.isna()'))
    + geom_point(data=summaries.query('type.notna()'),color='#F21A00', size=3)
    + geom_label(
        aes(label='type'), adjust_text={'arrowprops': {'arrowstyle':'-'}}
    )
    + scale_x_continuous(labels=percent_format())
    + scale_y_continuous(labels=percent_format())
    + labs(
        x='Volatility', y='Expected Return', title='Expected Return vs. Volatility in Efficient Frontier'
    )
)
summaries_figure.show()

rfr = 0.041 #this is based on the current yield of government 10y bonds
#here we use the Sharpe Ratio to calculate the expected return for the risk level
#it actually breaks here because the risk-free rate is above our highest expected return

summaries = (summaries
    .assign(ratio = lambda x: (x['mu_annualized']-rfr)/x['sigma_annualized'])
    .dropna(subset=['omega'])
    .query('0 < a < 1')
    .sort_values(by='ratio', ascending=False)
    .reset_index(drop=True)
    .drop(columns=['symbol','mu','sigma','omega_mvp','type'])
)
print(summaries)

In [ ]:
#3: analyze the effect of the correlation coefficient of assets' returns on the efficient frontier
#instructed to use hypothetical data, instead i will manipulate the correlation coefficient of existing stocks
!pip install -q tidyfinance
!pip install -q adjustText

import sys
print(sys.version)

import pandas as pd
import numpy as np
import tidyfinance as tf

from plotnine import *
from mizani.formatters import percent_format
from adjustText import adjust_text

symbols = ['AAPL', 'NVDA', 'TSLA', 'AMZN']

prices_daily = tf.download_data(
    domain = 'stock_prices',
    symbols = symbols,
    start_date = '2020-01-01',
    end_date = '2025-12-31'
)

prices_daily = (
    prices_daily
    .groupby('symbol')
    .apply(lambda x: x.assign(counts = lambda x: x['adjusted_close'].dropna().count()))
    .reset_index(drop=True)
    .query('counts == counts.max()')
)

returns = (
    prices_daily
    .assign(
        date = lambda x: x['date'].dt.to_period('M').dt.to_timestamp()
    )
    .reset_index(drop=True)
    .groupby(['date','symbol'], as_index=True)
    .aggregate(adjusted_close = ('adjusted_close','last'))
    .assign(ret=lambda x: x.groupby('symbol')['adjusted_close'].pct_change())
)



assets = (
    returns
    .groupby('symbol', as_index = False)
    .agg(
        mu = ('ret', 'mean'),
        sigma = ('ret', 'std')
    )
    .assign(
        mu_annualized = lambda x: (1+x['mu'])**12 - 1,
        sigma_annualized = lambda x: x['sigma']*np.sqrt(12)
    )
)

returns_wide = (
    returns
    .reset_index()
    .pivot(index = 'date', columns = 'symbol', values = 'ret')
)

multiplier = int(input('Enter a number to alter the covariance:'))

sigma = returns_wide.cov() * multiplier

sigma_long = (
    sigma
    .reset_index()
    .melt(id_vars='symbol', var_name='symbol_b', value_name = 'value')
)

sigma_long['symbol_b'] = pd.Categorical(
    sigma_long['symbol_b'],
    categories = sigma_long['symbol_b'].unique()[::-1],
    ordered=True
)

iota = np.ones(sigma.shape[0])
sigma_inv = np.linalg.inv(sigma.values)
omega_mvp = (sigma_inv @ iota) / (iota @ sigma_inv @ iota)

assets = assets.assign(omega_mvp=omega_mvp)

assets['symbol'] = pd.Categorical(
    assets['symbol'],
    categories = assets.sort_values('omega_mvp')['symbol'],
    ordered = True
)

mu = assets['mu_annualized'].values
mu_mvp = omega_mvp @ mu
sigma_mvp = np.sqrt(omega_mvp @ sigma.values @ omega_mvp)

summary_mvp = pd.DataFrame(
    {'mu_annualized': [mu_mvp], 'sigma_annualized': ['sigma_mvp'], 'type': 'Minimum-variance Portfolio'}
)

mu_bar = assets['mu_annualized'].max()

C = iota @ sigma_inv @ iota
D = iota @ sigma_inv @ mu
E = mu @ sigma_inv @ mu
lambda_tilde = 2 * (mu_bar - D / C) / (E - (D**2) / C)
omega_efp = omega_mvp + (lambda_tilde / 2) * (sigma_inv @ mu - D * omega_mvp)
mu_efp = omega_efp @ mu
sigma_efp = np.sqrt(omega_efp @ sigma.values @ omega_efp)

summary_efp = pd.DataFrame(
    {'mu_annualized': ['mu_efp'], 'sigma_annualized': ['sigma_efp'], 'type': ['Efficient Portfolio']}
)

summaries = pd.concat(
    ['assets', 'summary_mvp', 'summary_efp'], ignore_index = True
)